In [8]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chat_models import init_chat_model
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv

load_dotenv()
# ===============一、构建知识库阶段====================
# =============1.加载文档===============
loader = PyPDFLoader(file_path="resources/中华人民共和国劳动法_20181229.pdf", mode="single")
docs = loader.load()

# =============2.切分文档===============
# 创建递归切分器
splitter = CharacterTextSplitter(separator="\n", chunk_size=800, chunk_overlap=150)
# 切分文档
chunks = splitter.split_documents(docs)
print(f"分块数量：{len(chunks)}")

# =============3.向量化===============
# 向量模型，这里用阿里云的向量模型
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)
# =============4.存入向量库===============
# 创建向量库，需要指定向量模型，存储文档时会自动调用向量模型完成向量化
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

# ==============二、在线问答阶段================
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


def try_rag(query: str):
    # 1.检索文档（VectorStore会自动把问题向量化，召回相关知识片段）
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    # 2.拼接上下文提示词
    content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"""你基于我提供的报告回答用户问题，报告中没提及的就说不知道，不要自己编造答案.
    report: ```{content}```
    query: {query}"""
    # 3.调用模型，生成答案
    response = model.invoke(prompt)
    return response.content


# ================三、测试=================
print(try_rag("招用未成年人会怎么样"))
print('=' * 100)
# print(try_rag("茅台2025年的市盈率和市净率是多少"))

分块数量：15
根据您提供的报告内容，**未提及**招用未成年人会有什么后果或处罚规定。报告中仅提到“禁止用人单位招用未满十六周岁的未成年人”，并未说明违反该规定的法律责任。
